# 1. Read all CSV, import into single dataframe

In [1]:
import os
import pandas as pd
import numpy as np
from lets_plot import *
LetsPlot.setup_html()

# Identify the location of the original files
# This represents the path: ../data/waitrose-2024-07
data_folder = os.path.join('..', 'data', 'waitrose-2024-07')

# Use a list comprehension to get all the files in the folder
all_files = [os.path.join(data_folder, file) for file in os.listdir(data_folder) 
             if file.endswith('.csv')]

# Print the list of files if you want to check
print(all_files)

# Read every single file as a DataFrame
# Save the dataframes in a list
list_of_dfs = [pd.read_csv(file) for file in all_files]

# Use pd.concat to concatenate all the files into a single DataFrame
df = pd.concat(list_of_dfs)

# Check that we have all the data
df.info()

['../data/waitrose-2024-07/home.csv', '../data/waitrose-2024-07/best-of-british.csv', '../data/waitrose-2024-07/beer-wine-and-spirits.csv', '../data/waitrose-2024-07/everyday-value.csv', '../data/waitrose-2024-07/household.csv', '../data/waitrose-2024-07/pet.csv', '../data/waitrose-2024-07/food-cupboard.csv', '../data/waitrose-2024-07/dietary-and-lifestyle.csv', '../data/waitrose-2024-07/tea-coffee-and-soft-drinks.csv', '../data/waitrose-2024-07/frozen.csv', '../data/waitrose-2024-07/waitrose-brands.csv', '../data/waitrose-2024-07/summer.csv', '../data/waitrose-2024-07/fresh-and-chilled.csv', '../data/waitrose-2024-07/organic-shop.csv', '../data/waitrose-2024-07/bakery.csv', '../data/waitrose-2024-07/toiletries-health-and-beauty.csv', '../data/waitrose-2024-07/baby-child-and-parent.csv', '../data/waitrose-2024-07/new.csv']
<class 'pandas.core.frame.DataFrame'>
Index: 25418 entries, 0 to 805
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------

# Exploring "item-price" column

**observations**

- Some numbers are in format "£10.00"
- Others are in format "99p"
- Others are in format "99p each est."
- Others are in format "10p - 50p"
- Some rows don't have any prices at all

## Procedure

0) Remove all rows with NaN in "data-price" column
1) Remove pound symbol
2) if p, remove p and add 0. at the front of string
3) Remove each est.
4) Separate boundaries of range by -, average.
5) Sum whole column to check.

In [2]:
df = df.dropna(subset="item-price")
def clean_item_price(item_price:str):
    try:
        if '£' in item_price:
            item_price = str(item_price).replace("£","").strip()
        if 'p' in item_price:
            item_price = str(item_price).replace("p","").strip()
            item_price = "0." + item_price
        if item_price.endswith('each est.'):
            item_price = str(item_price).replace("each est.","").strip()
        if "-" in item_price:
            minimum, maximum = item_price.split("-")
            item_price = (float(minimum) + float(maximum))/2
        return float(item_price)
        
    except Exception as e:
        print(item_price, '---', e)

In [3]:
df["item-price"] = df["item-price"].apply(clean_item_price)

In [4]:
df["item-price"].astype(float).isna().sum()/len(df)

0.0

In [5]:
plot_df = (
    df.groupby('category')['item-price'].describe()
        .reset_index()
        .rename(columns={'25%': 'Q1', '50%': 'median', '75%': 'Q3'})
        .sort_values(by='median')
)

plot_df.head()
# plot_df.head() to see how it looks like

# This configures what shows up when you hover your mouse over the plot.
tooltip_setup = (
    layer_tooltips()
        .line('@category')
        .line('[@Q1 -- @median -- @Q3]')
        .format('@Q1', '£ {.2f}')
        .format('@median', '£ {.2f}')
        .format('@Q3', '£ {.2f}')
)

g = (
    # Maps the columns to the aesthetics of the plot.
    ggplot(plot_df, aes(y='category', x='median', xmin='Q1', xmax='Q3', fill='category')) +

    # GEOMS

    # Add a line range that 'listens to' columns informed in `ymin` and `ymax` aesthetics
    geom_linerange(size=1, alpha=0.75, tooltips=tooltip_setup) +

    # Add points to the plot (listen to `x` and `y` and fill aesthetics)
    geom_point(size=3, stroke=1, shape=21, tooltips=tooltip_setup) +

    # SCALES

    # Remove the legend (we can already read the categories from the y-axis)
    scale_fill_discrete(guide='none') +

    # Specify names for the axes
    scale_y_continuous(name="Categories\n(from largest to smallest median)", expand=[0.05, 0.05]) +
    scale_x_continuous(name="Price (£)", expand=[0., 0.05], format='£ {.2f}', breaks=np.arange(0, 20, 2.5)) +

    # LABELS
    # It's nice when the plot tells you the key takeaways
    labs(title='"Beer, Wine & Spirits" has the highest median price for individual items',
         subtitle="Dots represent the median price, bars represent the 25th and 75th percentiles") +
    theme(axis_text_x=element_text(size=15),
        axis_text_y=element_text(size=17),
        axis_title_x=element_text(size=20),
        axis_title_y=element_text(size=20),
        plot_title=element_text(size=19, face='bold'),
        plot_subtitle=element_text(size=18),
        legend_position='none') +
    ggsize(1000, 500)

)

g